# Classical Machine Learning Baselines & Out-Of-Fold Ensembling

Demonstrates training and evaluation of:
1. **Classical Baselines**: L2 Logistic Regression, ExtraTrees, and Gradient Boosted Trees (XGBoost).
2. **Out-of-Fold SLSQP Probability Blending**: Computes optimal convex weights $w_i \ge 0, \sum w_i = 1$ on validation sets (never test sets).
3. **Test-Time Augmentation (TTA)**: Multi-view inference for uncertainty quantification.


In [ ]:
# Cell 1: Classical baselines imports
import os
import sys
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.classical_baselines import (
    train_logistic_regression,
    train_extra_trees,
    train_single_fold_xgboost,
)
from src.models.ensemble import optimize_ensemble_weights, blend_predictions
from src.evaluation.metrics import compute_comprehensive_metrics

print("Classical baselines and ensemble modules loaded.")


## 1. Training Classical Baselines


In [ ]:
# Cell 2: Train baseline models on synthetic facial features
np.random.seed(42)
N_samples = 400
N_features = 39
X_train = np.random.randn(N_samples, N_features).astype(np.float32)
y_train = (X_train[:, 0] * 0.5 + X_train[:, 1] * 0.3 + np.random.randn(N_samples) * 0.1 > 0).astype(int)

lr_model = train_logistic_regression(X_train, y_train)
et_model = train_extra_trees(X_train, y_train, n_estimators=50)
xgb_model = train_single_fold_xgboost(X_train, y_train)

print("Baseline models successfully trained:")
print(f"  • Logistic Regression: {type(lr_model).__name__}")
print(f"  • ExtraTrees: {type(et_model).__name__}")
print(f"  • XGBoost / Gradient Boosting: {type(xgb_model).__name__}")


## 2. SLSQP Out-Of-Fold Ensembling

Find optimal blending weights minimizing log-loss on validation data.


In [ ]:
# Cell 3: Optimize ensemble weights
X_val = np.random.randn(100, N_features).astype(np.float32)
y_val = (X_val[:, 0] * 0.5 + X_val[:, 1] * 0.3 + np.random.randn(100) * 0.1 > 0).astype(int)

p_lr = lr_model.predict_proba(X_val)[:, 1]
p_et = et_model.predict_proba(X_val)[:, 1]
p_xgb = xgb_model.predict_proba(X_val)[:, 1]

weights = optimize_ensemble_weights([p_lr, p_et, p_xgb], y_val)
print(f"Optimized SLSQP Ensemble Weights: LR={weights[0]:.3f}, ET={weights[1]:.3f}, XGB={weights[2]:.3f}")

blended_val_probs = blend_predictions([p_lr, p_et, p_xgb], weights)
metrics = compute_comprehensive_metrics(y_val, blended_val_probs)
print(f"Blended Validation Accuracy: {metrics['accuracy']:.2%}")
print(f"Blended Validation F1-Score: {metrics['f1_score']:.4f}")
